# 15 Sensitivitas Praproses


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import io, itertools, json, os, sys, zipfile
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next(
    (p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()),
    None
)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')

LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('15_Sensitivitas_Praproses')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)
K = el.K
BASE_SEED = 4161600
DEGENERATE_MAX_SHARE = 0.95

df_frozen, X_frozen, FEATURES = el.load_lms_dataset(PAPER_ROOT)
df_frozen = df_frozen.copy()
df_frozen['user'] = df_frozen['user'].astype(str)

SEMANTIC_REFERENCE = artifact_path('08_theory_guided_pseudolabeled_dataset.csv','15_Sensitivitas_Praproses')
EVIDENCE_FILE = artifact_path('08_semantic_evidence_matrix.csv','15_Sensitivitas_Praproses')

if not SEMANTIC_REFERENCE.is_file() or not EVIDENCE_FILE.is_file():
    raise FileNotFoundError(
        'Output Notebook 08 belum lengkap. Jalankan 11_Interpretasi_Semantik_FSLSM.ipynb.'
    )

semantic_reference = pd.read_csv(SEMANTIC_REFERENCE, dtype={'user':str})
evidence = pd.read_csv(EVIDENCE_FILE)

semantic_reference = (
    semantic_reference.set_index('user')
    .loc[df_frozen['user']]
    .reset_index()
)

print('PAPER_ROOT:', PAPER_ROOT)
print('Rows:', len(df_frozen))
print('Features:', len(FEATURES))

FEATURE_PLAN = [
    {'feature':'content_visit','threshold_mode':'GRAF_FIXED','graf_low':75.0,'graf_high':100.0},
    {'feature':'content_stay','threshold_mode':'EMPIRICAL_POSITIVE_Q33_Q67','graf_low':50.0,'graf_high':75.0},
    {'feature':'selfass_visit_proxy','threshold_mode':'GRAF_FIXED','graf_low':25.0,'graf_high':75.0},
    {'feature':'selfass_stay_proxy','threshold_mode':'EMPIRICAL_POSITIVE_Q33_Q67','graf_low':50.0,'graf_high':75.0},
    {'feature':'exercise_visit_proxy','threshold_mode':'GRAF_FIXED','graf_low':25.0,'graf_high':75.0},
    {'feature':'exercise_stay_proxy','threshold_mode':'EMPIRICAL_POSITIVE_Q33_Q67','graf_low':50.0,'graf_high':75.0},
    {'feature':'forum_visit','threshold_mode':'GRAF_WITH_DEGENERACY_FALLBACK','graf_low':7.0,'graf_high':50.0},
    {'feature':'forum_stay','threshold_mode':'GRAF_WITH_DEGENERACY_FALLBACK','graf_low':5.0,'graf_high':30.0},
    {'feature':'forum_post','threshold_mode':'GRAF_WITH_SPARSE_FALLBACK','graf_low':1.0,'graf_high':10.0},
    {'feature':'quiz_stay_results_proxy','threshold_mode':'GRAF_FIXED','graf_low':30.0,'graf_high':60.0},
    {'feature':'navigation_overview_visit','threshold_mode':'GRAF_WITH_DEGENERACY_FALLBACK','graf_low':10.0,'graf_high':20.0},
    {'feature':'navigation_overview_stay','threshold_mode':'EMPIRICAL_POSITIVE_Q33_Q67','graf_low':50.0,'graf_high':75.0},
    {'feature':'return_to_overview_rate','threshold_mode':'EMPIRICAL_POSITIVE_Q33_Q67','graf_low':np.nan,'graf_high':np.nan},
]

if [x['feature'] for x in FEATURE_PLAN] != FEATURES:
    raise RuntimeError('Feature plan tidak sama dengan frozen feature order.')


def normalize_user_column(frame):
    frame = frame.copy()

    # Current V4.3/frozen files use "user".
    if 'user' not in frame.columns:
        # Some older exports use userid.
        if 'userid' in frame.columns:
            frame = frame.rename(columns={'userid':'user'})
        else:
            return frame

    frame['user'] = frame['user'].astype(str)
    return frame


def valid_numeric_source(frame):
    frame = normalize_user_column(frame)
    return 'user' in frame.columns and set(FEATURES).issubset(frame.columns)


def align_to_frozen_users(frame, source_name):
    frame = normalize_user_column(frame)

    if not valid_numeric_source(frame):
        return None

    # A selected-user-course table should already be one row per learner.
    # If a broader table is encountered, duplicate users are not silently resolved.
    if frame['user'].duplicated().any():
        return None

    frozen_users = df_frozen['user'].astype(str).tolist()
    available = set(frame['user'])

    if not set(frozen_users).issubset(available):
        return None

    aligned = (
        frame.set_index('user')
        .loc[frozen_users, FEATURES]
        .reset_index()
    )

    for f in FEATURES:
        aligned[f] = pd.to_numeric(
            aligned[f],
            errors='coerce',
        ).fillna(0.0)

    return aligned, source_name


def try_csv(path):
    try:
        frame = pd.read_csv(
            path,
            dtype={'user':str, 'userid':str},
            low_memory=False,
        )
    except Exception:
        return None

    return align_to_frozen_users(frame, str(path))


def resolve_numeric_source():
    path=lms_data('numeric')
    if not path.is_file():raise FileNotFoundError(f'Input numerik final belum tersedia: {path}')
    result=try_csv(path)
    if result is None:raise RuntimeError(f'Input numerik final tidak dapat disejajarkan dengan cohort LMS: {path}')
    return result


numeric_raw, NUMERIC_SOURCE = resolve_numeric_source()

# Input final disejajarkan menurut user tanpa membuat salinan data.
numeric = numeric_raw.copy()

if numeric['user'].duplicated().any():
    raise RuntimeError('Numeric source memiliki user duplikat setelah alignment.')

if numeric['user'].astype(str).tolist() != df_frozen['user'].astype(str).tolist():
    raise RuntimeError('Urutan user numeric source tidak identik dengan frozen CARE-KM dataset.')

for f in FEATURES:
    numeric[f] = pd.to_numeric(
        numeric[f],
        errors='coerce',
    ).fillna(0.0)

print('Numeric source:', NUMERIC_SOURCE)
print('Aligned numeric rows:', len(numeric))

def classify_two_cut(series, low, high):
    x = pd.to_numeric(series, errors='coerce').fillna(0.0).astype(float)
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        raise ValueError(f'Invalid threshold {low}, {high}')
    return pd.Series(
        np.where(x < low, 'L', np.where(x < high, 'M', 'H')),
        index=series.index,
        dtype='object'
    )


def category_share(cat):
    return cat.value_counts(normalize=True).reindex(['L','M','H'], fill_value=0.0)


def positive_quantile_thresholds(series, q1, q2):
    x = pd.to_numeric(series, errors='coerce').fillna(0.0).astype(float)
    pos = x[x > 0]
    if pos.empty:
        return np.nan, np.nan

    low = float(pos.quantile(q1))
    high = float(pos.quantile(q2))

    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        uniq = np.sort(pos.unique())
        if len(uniq) < 2:
            return float(uniq[0]), float(uniq[0])
        low = float(np.quantile(uniq, q1))
        high = float(np.quantile(uniq, q2))

    return low, high


def sparse_zero_median_categories(series):
    x = pd.to_numeric(series, errors='coerce').fillna(0.0).astype(float)
    pos = x[x > 0]
    if pos.empty:
        return pd.Series('L', index=series.index, dtype='object'), 0.0, np.nan
    med = float(pos.median())
    cat = pd.Series(
        np.where(x <= 0, 'L', np.where(x < med, 'M', 'H')),
        index=series.index,
        dtype='object'
    )
    return cat, 0.0, med


def build_categories(numeric_df, q1=1/3, q2=2/3, graf_scale=1.0):
    cat = pd.DataFrame({'user':numeric_df['user'].astype(str)})
    audit = []

    for spec in FEATURE_PLAN:
        f = spec['feature']
        mode = spec['threshold_mode']
        x = numeric_df[f]

        gl = spec['graf_low']
        gh = spec['graf_high']
        gl = float(gl * graf_scale) if np.isfinite(gl) else np.nan
        gh = float(gh * graf_scale) if np.isfinite(gh) else np.nan

        low = high = np.nan
        fallback = False

        if mode == 'GRAF_FIXED':
            c = classify_two_cut(x, gl, gh)
            low, high = gl, gh
            applied = 'GRAF_FIXED'

        elif mode == 'EMPIRICAL_POSITIVE_Q33_Q67':
            low, high = positive_quantile_thresholds(x, q1, q2)
            if not np.isfinite(low) or not np.isfinite(high) or high <= low:
                raise RuntimeError(f'Quantile threshold gagal untuk {f}.')
            c = classify_two_cut(x, low, high)
            applied = f'POSITIVE_Q{q1:.4f}_Q{q2:.4f}'

        elif mode == 'GRAF_WITH_DEGENERACY_FALLBACK':
            c0 = classify_two_cut(x, gl, gh)
            sh = category_share(c0)
            deg = float(sh.max()) > DEGENERATE_MAX_SHARE or int((sh == 0).sum()) > 0
            if deg:
                low, high = positive_quantile_thresholds(x, q1, q2)
                if not np.isfinite(low) or not np.isfinite(high) or high <= low:
                    raise RuntimeError(f'Fallback gagal untuk {f}.')
                c = classify_two_cut(x, low, high)
                applied = f'GRAF_FALLBACK_POSITIVE_Q{q1:.4f}_Q{q2:.4f}'
                fallback = True
            else:
                c = c0
                low, high = gl, gh
                applied = 'GRAF_FIXED_TRANSFERABLE'

        elif mode == 'GRAF_WITH_SPARSE_FALLBACK':
            c0 = classify_two_cut(x, gl, gh)
            sh = category_share(c0)
            deg = float(sh.max()) > DEGENERATE_MAX_SHARE or int((sh == 0).sum()) > 0
            if deg:
                c, low, high = sparse_zero_median_categories(x)
                applied = 'GRAF_FALLBACK_ZERO_AWARE_POSITIVE_MEDIAN'
                fallback = True
            else:
                c = c0
                low, high = gl, gh
                applied = 'GRAF_FIXED_TRANSFERABLE'
        else:
            raise ValueError(mode)

        cat[f] = c
        sh = category_share(c)
        audit.append({
            'feature':f,
            'applied_method':applied,
            'applied_low':low,
            'applied_high':high,
            'fallback_used':fallback,
            'L_share':float(sh['L']),
            'M_share':float(sh['M']),
            'H_share':float(sh['H']),
            'n_categories':int(c.nunique()),
        })

    scaled = pd.DataFrame({'user':cat['user']})
    for f in FEATURES:
        scaled[f] = cat[f].map({'L':'1','M':'2','H':'3'}).astype(str)

    return scaled, pd.DataFrame(audit)


CONFIGS = [
    {'name':'PRIMARY','q1':1/3,'q2':2/3,'graf_scale':1.00},
    {'name':'EMPIRICAL_Q30_Q70','q1':0.30,'q2':0.70,'graf_scale':1.00},
    {'name':'EMPIRICAL_Q35_Q65','q1':0.35,'q2':0.65,'graf_scale':1.00},
    {'name':'GRAF_MINUS_10PCT','q1':1/3,'q2':2/3,'graf_scale':0.90},
    {'name':'GRAF_PLUS_10PCT','q1':1/3,'q2':2/3,'graf_scale':1.10},
]

primary_rebuilt, _ = build_categories(numeric)
primary_X = primary_rebuilt[FEATURES].to_numpy(dtype=object)

mismatch = int((primary_X != X_frozen).sum())
if mismatch:
    raise RuntimeError(
        f'PRIMARY reconstruction tidak identik dengan frozen dataset: {mismatch} cell berbeda.'
    )

print('PRIMARY reconstruction: EXACT MATCH')

DIMENSIONS = [
    ('Active/Reflective', ('Active','Reflective')),
    ('Sensing/Intuitive', ('Sensing','Intuitive')),
    ('Visual/Verbal', ('Visual','Verbal')),
    ('Sequential/Global', ('Sequential','Global')),
]
POLES = {d:p for d,p in DIMENSIONS}
ABBR = {
    'Active':'Act.','Reflective':'Ref.',
    'Sensing':'Sens.','Intuitive':'Int.',
    'Visual':'Vis.','Verbal':'Ver.',
    'Sequential':'Seq.','Global':'Gl.'
}
DIM_KEYS = [
    'active_reflective',
    'sensing_intuitive',
    'visual_verbal',
    'sequential_global',
]
LS_LOOKUP = {
    tuple(combo):f'LS{i:02d}'
    for i, combo in enumerate(
        itertools.product(*[p for _,p in DIMENSIONS]), start=1
    )
}


def signed_vote(value, rule):
    value = int(value)
    first, second = POLES[str(rule.dimension)]
    if value == 2:
        return 0
    pole = str(rule.high_supports) if value == 3 else str(rule.low_supports)
    if pole == first:
        return 1
    if pole == second:
        return -1
    raise ValueError(pole)


def semantic_mapping(X, modes, labels):
    X = np.asarray(X, dtype=object)
    labels = np.asarray(labels, dtype=int)
    modes = np.asarray(modes, dtype=object)
    fpos = {f:i for i,f in enumerate(FEATURES)}
    rows = []

    for cid in range(K):
        selected, abbrs = [], []

        for dimension, (first, second) in DIMENSIONS:
            rules = evidence[evidence['dimension'].astype(str).eq(dimension)]

            mv = np.asarray([
                signed_vote(modes[cid, fpos[str(r.feature)]], r)
                for r in rules.itertuples(index=False)
            ])
            ms = int(np.sum(mv > 0) - np.sum(mv < 0))

            member_X = X[labels == cid]
            nv = []
            for r in rules.itertuples(index=False):
                j = fpos[str(r.feature)]
                nv.extend(signed_vote(v, r) for v in member_X[:, j])
            nv = np.asarray(nv)
            ns = int(np.sum(nv > 0) - np.sum(nv < 0))

            if ms > 0:
                pole = first
            elif ms < 0:
                pole = second
            elif ns > 0:
                pole = first
            elif ns < 0:
                pole = second
            else:
                pole = None

            selected.append(pole)
            abbrs.append('Balanced' if pole is None else ABBR[pole])

        unresolved = any(x is None for x in selected)
        rows.append({
            'cluster_id':cid,
            'active_reflective':abbrs[0],
            'sensing_intuitive':abbrs[1],
            'visual_verbal':abbrs[2],
            'sequential_global':abbrs[3],
            'ls_class':'AMBIGUOUS' if unresolved else LS_LOOKUP[tuple(selected)],
            'mapping_status':'BALANCED' if unresolved else 'RESOLVED',
        })

    return pd.DataFrame(rows)

summary_rows = []
threshold_frames = []
feature_change_rows = []
learner_frames = []

primary_labels = None
primary_categorical = None

for cfg in CONFIGS:
    name = cfg['name']
    print('\n===', name, '===')

    cat_df, audit = build_categories(
        numeric,
        q1=cfg['q1'],
        q2=cfg['q2'],
        graf_scale=cfg['graf_scale'],
    )
    audit.insert(0, 'configuration', name)
    threshold_frames.append(audit)

    X = cat_df[FEATURES].to_numpy(dtype=object)

    row, _, result = ck.run_method(
        'CARE-KM', X, None, K,
        seed=BASE_SEED,
        care_config=CARE_CONFIG,
        si_sample_size=el.SI_SAMPLE_SIZE,
        si_seed=BASE_SEED,
        max_iter=el.MAX_ITER,
    )
    labels = np.asarray(result['labels'], dtype=int)

    if name == 'PRIMARY':
        primary_labels = labels.copy()
        primary_categorical = X.copy()

        ref_file = artifact_path('04_carekm_reference_labels.csv','15_Sensitivitas_Praproses')
        if ref_file.is_file():
            ref = pd.read_csv(ref_file, dtype={'user':str})
            rm = ref.set_index('user')['cluster_id'].astype(int)
            ref_labels = df_frozen['user'].map(rm).to_numpy()
            if adjusted_rand_score(ref_labels, labels) < 1 - 1e-12:
                raise RuntimeError('PRIMARY partition berbeda dari Notebook 04 reference.')

    mapping = semantic_mapping(X, result['modes'], labels)
    mp = mapping.set_index('cluster_id')

    current = pd.DataFrame({
        'user':df_frozen['user'].astype(str),
        'configuration':name,
        'cluster_id':labels,
    })
    for c in DIM_KEYS + ['ls_class','mapping_status']:
        current[c] = [mp.loc[int(k), c] for k in labels]

    ref = semantic_reference[['user','ls_class',*DIM_KEYS]].copy()
    ref = ref.rename(columns={c:f'{c}_reference' for c in ['ls_class',*DIM_KEYS]})
    current = current.merge(ref, on='user', how='left', validate='one_to_one')

    for c in DIM_KEYS:
        current[f'{c}_agreement'] = (
            current[c].astype(str) == current[f'{c}_reference'].astype(str)
        )

    current['mean_dimension_agreement'] = current[
        [f'{c}_agreement' for c in DIM_KEYS]
    ].mean(axis=1)
    current['base_ls_agreement'] = (
        current['ls_class'].astype(str) == current['ls_class_reference'].astype(str)
    )
    learner_frames.append(current)

    changed = X != primary_categorical
    for j, f in enumerate(FEATURES):
        feature_change_rows.append({
            'configuration':name,
            'feature':f,
            'changed_cells':int(changed[:,j].sum()),
            'changed_share':float(changed[:,j].mean()),
        })

    summary_rows.append({
        'configuration':name,
        'q1':cfg['q1'],
        'q2':cfg['q2'],
        'graf_scale':cfg['graf_scale'],
        'initial_cost':float(row['initial_cost']),
        'final_cost':float(row['final_cost']),
        'dbi':float(row['dbi']),
        'si':float(row['si']),
        'iterations':int(row['iterations']),
        'ari_vs_primary_partition':float(adjusted_rand_score(primary_labels, labels)),
        'nmi_vs_primary_partition':float(normalized_mutual_info_score(primary_labels, labels)),
        'changed_categorical_cell_share':float(changed.mean()),
        'changed_learner_profile_share':float(np.any(changed, axis=1).mean()),
        'active_reflective_agreement':float(current['active_reflective_agreement'].mean()),
        'sensing_intuitive_agreement':float(current['sensing_intuitive_agreement'].mean()),
        'visual_verbal_agreement':float(current['visual_verbal_agreement'].mean()),
        'sequential_global_agreement':float(current['sequential_global_agreement'].mean()),
        'mean_dimensional_agreement':float(current['mean_dimension_agreement'].mean()),
        'base_ls_agreement':float(current['base_ls_agreement'].mean()),
        'unresolved_learner_rate':float(current['mapping_status'].ne('RESOLVED').mean()),
    })

summary = pd.DataFrame(summary_rows)
thresholds = pd.concat(threshold_frames, ignore_index=True)
feature_changes = pd.DataFrame(feature_change_rows)
learner_semantic = pd.concat(learner_frames, ignore_index=True)

el.save_csv(summary, artifact_path('preprocessing_sensitivity_summary.csv','15_Sensitivitas_Praproses'), index=False)
el.save_csv(thresholds, artifact_path('preprocessing_sensitivity_thresholds.csv','15_Sensitivitas_Praproses'), index=False)
el.save_csv(feature_changes, artifact_path('preprocessing_sensitivity_feature_changes.csv','15_Sensitivitas_Praproses'), index=False)
el.save_csv(learner_semantic, artifact_path('preprocessing_sensitivity_learner_semantic.csv','15_Sensitivitas_Praproses'), index=False)

display(summary)

validation = {
    'analysis':'preprocessing sensitivity',
    'primary_reconstruction_exact_match':True,
    'numeric_source':NUMERIC_SOURCE,
    'configurations':CONFIGS,
    'tuning_or_model_selection':False,
    'care_config':CARE_CONFIG,
    'K':K,
    'semantic_rule':'mode-first; member evidence only for exact mode tie',
    'outputs':[
        'preprocessing_sensitivity_summary.csv',
        'preprocessing_sensitivity_thresholds.csv',
        'preprocessing_sensitivity_feature_changes.csv',
        'preprocessing_sensitivity_learner_semantic.csv',
    ],
}
(artifact_path('preprocessing_sensitivity_validation.json','15_Sensitivitas_Praproses')).write_text(
    json.dumps(validation, indent=2), encoding='utf-8'
)
print(json.dumps(validation, indent=2))